# Phase 1 — Day 4: Faker & Python Core

**Date:** 2026-04-23

Today we learn how to generate realistic fake data with the Faker library and sharpen our core Python skills. These are tools you'll use constantly in data science for prototyping, testing, and writing clean code.

**Learning Objectives:**
- Generate realistic synthetic data using Faker
- Create random datasets with numpy.random
- Write concise list and dict comprehensions
- Build reusable functions with default arguments
- Handle errors gracefully with try/except

In [ ]:
# Setup - install Faker if needed, then import everything
# !pip install faker

from faker import Faker
import numpy as np
import pandas as pd
import random

# Set seeds for reproducibility
fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

print("All imports ready!")

---
## 1. Faker Library Basics

Faker generates realistic-looking fake data: names, addresses, emails, dates, credit card numbers, and hundreds more. It's perfect for creating test datasets without using real personal information.

The main idea is simple: create a `Faker()` instance, then call methods like `fake.name()`, `fake.email()`, etc. Each call returns a different random value. You can also set a seed for reproducibility.

In [ ]:
# Basic Faker usage
print("Name:       ", fake.name())
print("Email:      ", fake.email())
print("Address:    ", fake.address())
print("Phone:      ", fake.phone_number())
print("Company:    ", fake.company())
print("Job:        ", fake.job())
print("Date:       ", fake.date())
print("Text:       ", fake.text(max_nb_chars=60))

In [ ]:
# Build a fake customer dataset with Faker
customers = []
for _ in range(10):
    customers.append({
        "name": fake.name(),
        "email": fake.email(),
        "city": fake.city(),
        "signup_date": fake.date_between(start_date="-2y", end_date="today"),
        "age": fake.random_int(min=18, max=70)
    })

df_customers = pd.DataFrame(customers)
print(f"Generated {len(df_customers)} fake customers:")
df_customers

In [ ]:
# Faker with locales - generate data in different languages
fake_tr = Faker("tr_TR")  # Turkish
fake_de = Faker("de_DE")  # German

print("Turkish name:  ", fake_tr.name())
print("Turkish city:  ", fake_tr.city())
print("German name:   ", fake_de.name())
print("German address:", fake_de.address())

---
## 2. numpy.random for Synthetic Data

While Faker is great for realistic personal data, `numpy.random` is your go-to for generating numerical data. You can sample from distributions (normal, uniform, binomial, etc.) and create arrays of any shape.

This is essential when you need to simulate measurements, scores, prices, or any numeric column in a test dataset.

In [ ]:
# numpy.random basics
rng = np.random.default_rng(42)  # modern way to create a generator

# Different distributions
uniform_vals = rng.uniform(low=0, high=100, size=5)       # flat between 0-100
normal_vals = rng.normal(loc=50, scale=10, size=5)         # bell curve, mean=50, std=10
integer_vals = rng.integers(low=1, high=7, size=5)         # dice rolls (1-6)
choice_vals = rng.choice(["A", "B", "C"], size=5)          # random picks

print("Uniform:  ", uniform_vals.round(2))
print("Normal:   ", normal_vals.round(2))
print("Integers: ", integer_vals)
print("Choices:  ", choice_vals)

In [ ]:
# Combine Faker + numpy.random to build a realistic dataset
rng2 = np.random.default_rng(99)

records = []
for _ in range(200):
    records.append({
        "name": fake.name(),
        "salary": round(rng2.normal(loc=55000, scale=15000), 2),
        "department": rng2.choice(["Engineering", "Sales", "Marketing", "HR"]),
        "satisfaction_score": rng2.integers(1, 11),  # 1-10
        "years_at_company": rng2.integers(0, 20)
    })

df_employees = pd.DataFrame(records)
print(f"Shape: {df_employees.shape}")
print(df_employees.describe().round(2))
df_employees.head()

---
## 3. List Comprehensions

List comprehensions are a compact way to create lists. Instead of writing a for loop, appending to a list, you do it in one line. The syntax is `[expression for item in iterable if condition]`.

They're not just shorter. They're also faster than regular loops because Python optimizes them internally. You'll see them everywhere in data science code.

In [ ]:
# Regular loop vs list comprehension
# The old way
squares_loop = []
for x in range(10):
    squares_loop.append(x ** 2)

# The comprehension way
squares_comp = [x ** 2 for x in range(10)]

print("Loop:         ", squares_loop)
print("Comprehension:", squares_comp)
print("Same result?  ", squares_loop == squares_comp)

In [ ]:
# Comprehensions with conditions (filtering)
numbers = [1, -3, 5, -7, 9, -2, 4, 0]

# Only keep positives
positives = [n for n in numbers if n > 0]
print("Positives:", positives)

# Transform + filter: square only even numbers
even_squares = [n ** 2 for n in range(20) if n % 2 == 0]
print("Even squares:", even_squares)

# Nested comprehension: flatten a 2D list
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat = [val for row in matrix for val in row]
print("Flattened:", flat)

---
## 4. Dict Comprehensions

Same idea as list comprehensions, but they produce dictionaries. The syntax is `{key_expr: value_expr for item in iterable}`. Super handy for quick lookups, transforming data, and inverting mappings.

In [ ]:
# Dict comprehension basics
# Create a word -> length mapping
words = ["python", "data", "science", "faker", "numpy"]
word_lengths = {w: len(w) for w in words}
print("Word lengths:", word_lengths)

# Invert a dictionary (swap keys and values)
status_codes = {200: "OK", 404: "Not Found", 500: "Server Error"}
inverted = {v: k for k, v in status_codes.items()}
print("Inverted:", inverted)

# Filter a dict: only keep items where value > 4
long_words = {w: l for w, l in word_lengths.items() if l > 4}
print("Long words:", long_words)

In [ ]:
# Practical example: build a fake lookup table with Faker + dict comprehension
user_ids = list(range(1001, 1011))
user_directory = {uid: {"name": fake.name(), "email": fake.email()} for uid in user_ids}

# Now you can look up any user instantly
print("User 1005:", user_directory[1005])
print()
for uid, info in list(user_directory.items())[:3]:
    print(f"  ID {uid}: {info['name']} ({info['email']})")

---
## 5. Functions

Functions let you package reusable logic. In data science, you'll write functions for data cleaning steps, feature engineering, custom metrics, and more.

Key things to remember: use default arguments to make functions flexible, add type hints for clarity, and keep each function focused on one task.

In [ ]:
# Functions with default arguments and type hints
def generate_fake_dataset(n_rows: int = 50, seed: int = 42) -> pd.DataFrame:
    """Generate a fake customer dataset with n_rows records."""
    Faker.seed(seed)
    local_rng = np.random.default_rng(seed)
    local_fake = Faker()
    
    data = {
        "customer_id": list(range(1, n_rows + 1)),
        "name": [local_fake.name() for _ in range(n_rows)],
        "email": [local_fake.email() for _ in range(n_rows)],
        "age": local_rng.integers(18, 65, size=n_rows),
        "purchase_amount": local_rng.normal(100, 30, size=n_rows).round(2)
    }
    return pd.DataFrame(data)

# Use with defaults
df_small = generate_fake_dataset()
print(f"Default: {df_small.shape}")

# Override arguments
df_large = generate_fake_dataset(n_rows=1000, seed=99)
print(f"Custom:  {df_large.shape}")
df_small.head()

In [ ]:
# Lambda functions: one-line throwaway functions
# Great for quick transforms in pandas

df_small["name_upper"] = df_small["name"].apply(lambda x: x.upper())
df_small["is_high_spender"] = df_small["purchase_amount"].apply(lambda x: x > 120)

print("High spenders:")
print(df_small[df_small["is_high_spender"]][["name", "purchase_amount"]].head())

# You can also use lambdas with sorted()
names = ["Charlie", "alice", "Bob"]
sorted_names = sorted(names, key=lambda x: x.lower())
print("\nSorted (case-insensitive):", sorted_names)

In [ ]:
# *args and **kwargs: flexible function signatures
def describe_data(df: pd.DataFrame, *columns, **options):
    """Print stats for selected columns with options."""
    round_to = options.get("round_to", 2)
    show_dtypes = options.get("show_dtypes", False)
    
    cols = columns if columns else df.select_dtypes(include="number").columns
    
    for col in cols:
        if col in df.columns:
            print(f"  {col}: mean={df[col].mean():.{round_to}f}, std={df[col].std():.{round_to}f}")
    
    if show_dtypes:
        print("\nDtypes:", dict(df[list(cols)].dtypes))

# Call with different argument combos
print("All numeric columns:")
describe_data(df_small)

print("\nJust age, with 4 decimal places:")
describe_data(df_small, "age", round_to=4, show_dtypes=True)

---
## 6. Error Handling with try/except

Real-world data is messy. Files might be missing, values might be the wrong type, APIs might time out. `try/except` lets your code handle these problems gracefully instead of crashing.

The basic pattern: put risky code in `try`, handle specific errors in `except`, and optionally use `finally` for cleanup that should always happen.

In [ ]:
# Basic try/except
def safe_divide(a, b):
    try:
        result = a / b
        return result
    except ZeroDivisionError:
        print(f"Warning: can't divide {a} by zero, returning None")
        return None
    except TypeError as e:
        print(f"Warning: wrong types given - {e}")
        return None

print(safe_divide(10, 3))      # works fine
print(safe_divide(10, 0))      # catches ZeroDivisionError
print(safe_divide("10", 3))    # catches TypeError

In [ ]:
# Real-world pattern: safely parse messy data
raw_values = ["42", "3.14", "N/A", "", "100", "abc", "7"]

def safe_float(val):
    """Convert a string to float, return None if it fails."""
    try:
        return float(val)
    except (ValueError, TypeError):
        return None

cleaned = [safe_float(v) for v in raw_values]
print("Raw:    ", raw_values)
print("Cleaned:", cleaned)

# Filter out the Nones
valid = [x for x in cleaned if x is not None]
print("Valid:  ", valid)
print(f"Mean of valid values: {np.mean(valid):.2f}")

In [ ]:
# try/except/else/finally - the full pattern
def load_data(filename):
    """Demonstrate the full try/except/else/finally pattern."""
    print(f"Attempting to load '{filename}'...")
    try:
        # This would normally be pd.read_csv(filename)
        if filename == "bad_file.csv":
            raise FileNotFoundError(f"No such file: '{filename}'")
        data = {"rows": 100, "cols": 5}  # simulated success
    except FileNotFoundError as e:
        print(f"  ERROR: {e}")
        data = None
    else:
        # Only runs if try succeeded (no exception)
        print(f"  SUCCESS: loaded {data['rows']} rows, {data['cols']} columns")
    finally:
        # Always runs, no matter what
        print(f"  DONE: finished processing '{filename}'")
    return data

result1 = load_data("good_file.csv")
print()
result2 = load_data("bad_file.csv")

---
## Tricky Bits

These are common mistakes people make with the concepts we covered today. Read the code, predict what happens, then run it.

In [ ]:
# TRAP 1: Mutable default arguments
# This is one of Python's most famous gotchas

def add_item(item, items=[]):
    """This function has a bug. Can you spot it?"""
    items.append(item)
    return items

# Watch what happens when you call it multiple times:
print("Call 1:", add_item("apple"))
print("Call 2:", add_item("banana"))
print("Call 3:", add_item("cherry"))
# Expected: each call returns a list with one item
# Actual: the default list is SHARED across calls!

# The fix: use None as default
def add_item_fixed(item, items=None):
    if items is None:
        items = []
    items.append(item)
    return items

print("\nFixed:")
print("Call 1:", add_item_fixed("apple"))
print("Call 2:", add_item_fixed("banana"))

In [ ]:
# TRAP 2: Catching too broadly
# Never use bare 'except:' - it hides bugs

try:
    # Imagine a typo: 'pritn' instead of 'print'
    result = int("hello")
except:  # BAD: catches EVERYTHING, even KeyboardInterrupt
    print("Something went wrong (but what?)")

# Better: catch specific exceptions
try:
    result = int("hello")
except ValueError as e:
    print(f"Caught ValueError: {e}")

# TRAP 3: Faker without seed gives different results each time
f1, f2 = Faker(), Faker()
# Without seed, these will differ
print(f"\nNo seed: '{f1.name()}' vs '{f2.name()}'")

# With seed, they match
Faker.seed(0)
f3 = Faker()
name_a = f3.name()
Faker.seed(0)
f4 = Faker()
name_b = f4.name()
print(f"With seed: '{name_a}' vs '{name_b}' -> same? {name_a == name_b}")

---
## Trick Questions

Test your understanding. Try to answer before revealing the solution.

**Q1:** What happens if you call `fake.name()` twice without setting a seed between calls?

<details><summary>Answer</summary>
You get two different names. Faker generates a new random value each time you call a method. If you want reproducible results, call `Faker.seed(n)` before generating data.
</details>

**Q2:** What's the difference between `[x for x in range(5)]` and `(x for x in range(5))`?

<details><summary>Answer</summary>
Square brackets create a list (all values computed immediately and stored in memory). Round brackets create a generator (values computed lazily, one at a time). For large datasets, generators use much less memory.
</details>

**Q3:** Why does `{1: "a", 1: "b"}` only have one entry?

<details><summary>Answer</summary>
Dictionary keys must be unique. When you use the same key twice, the second value overwrites the first. So you get `{1: "b"}`.
</details>

**Q4:** What does `except Exception as e` catch that `except:` (bare except) also catches?

<details><summary>Answer</summary>
`except Exception` catches all "normal" exceptions (ValueError, TypeError, etc.) but NOT system-exiting exceptions like KeyboardInterrupt and SystemExit. A bare `except:` catches absolutely everything, which is dangerous because it can prevent you from stopping a runaway script with Ctrl+C.
</details>

**Q5:** If you write `scores = {s: s**2 for s in [1, 2, 2, 3]}`, how many keys does `scores` have?

<details><summary>Answer</summary>
Three keys: `{1: 1, 2: 4, 3: 9}`. The duplicate `2` in the input list just means the key `2` gets set twice (both times to `4`), so the final dict has 3 unique keys.
</details>

---
## Exercises

Fill in the `___` blanks to make each cell work. The `assert` statements will confirm you got it right.

In [ ]:
# Exercise 1: Use a list comprehension to get all names that start with "J"
# from the fake customer dataframe we created earlier.

names = df_customers["name"].tolist()
j_names = [name for name in names if name.startswith(___)]

print("J names:", j_names)
assert all(n.startswith("J") for n in j_names), "All names should start with J"
print("Passed!")

In [ ]:
# Exercise 2: Create a dict comprehension that maps each number to its cube.
# Input: numbers 1 through 5. Output: {1: 1, 2: 8, 3: 27, 4: 64, 5: 125}

cubes = {n: ___ for n in range(1, 6)}

assert cubes == {1: 1, 2: 8, 3: 27, 4: 64, 5: 125}
print("Cubes:", cubes)
print("Passed!")

In [ ]:
# Exercise 3: Write a function that generates n fake email addresses using Faker.
# It should accept a seed parameter for reproducibility.

def generate_emails(n: int, seed: int = 0) -> list:
    Faker.seed(seed)
    f = Faker()
    return [___ for _ in range(n)]

emails = generate_emails(5, seed=42)
assert len(emails) == 5
assert all("@" in e for e in emails), "Each should be a valid email"
print("Emails:", emails)
print("Passed!")

In [ ]:
# Exercise 4: Use numpy.random to generate 100 random ages between 18 and 65.
# Then calculate the mean age.

rng_ex = np.random.default_rng(42)
ages = rng_ex.integers(low=___, high=___, size=___)

assert len(ages) == 100
assert ages.min() >= 18
assert ages.max() <= 65
print(f"Mean age: {ages.mean():.1f}")
print("Passed!")

In [ ]:
# Exercise 5: Write a safe_int function that converts a string to int.
# Return -1 if the conversion fails.

def safe_int(val):
    try:
        return ___(val)
    except ___:
        return -1

assert safe_int("42") == 42
assert safe_int("hello") == -1
assert safe_int("3.14") == -1  # int() can't parse floats directly
assert safe_int("") == -1
print("All assertions passed!")

In [ ]:
# Exercise 6: Use Faker + list comprehension to create a list of 20 fake 
# (name, city) tuples. Then convert it to a dict.

Faker.seed(123)
f_ex = Faker()
name_city_pairs = [(___, ___) for _ in range(20)]
name_city_dict = dict(name_city_pairs)

assert len(name_city_pairs) == 20
assert isinstance(name_city_pairs[0], tuple)
assert isinstance(name_city_dict, dict)
print(f"Got {len(name_city_dict)} unique names mapped to cities")
print("Sample:", list(name_city_dict.items())[:3])
print("Passed!")

In [ ]:
# Exercise 7: Write a function that takes a list of mixed values and returns
# only the ones that can be converted to float, as floats.
# Use try/except inside a list comprehension (yes, you'll need a helper function).

def extract_numbers(values):
    def try_float(v):
        try:
            return float(v)
        except (ValueError, TypeError):
            return None
    return [___ for v in values if ___ is not None]

test_input = [1, "3.14", "hello", None, "42", "", 7.5, "N/A"]
result = extract_numbers(test_input)

assert result == [1.0, 3.14, 42.0, 7.5]
print("Result:", result)
print("Passed!")

### Exercise Solutions

<details><summary>Click to reveal all solutions</summary>

**Exercise 1:** `"J"`

**Exercise 2:** `n ** 3`

**Exercise 3:** `f.email()`

**Exercise 4:** `low=18, high=66, size=100` (high is exclusive in `rng.integers`)

**Exercise 5:** `int` and `ValueError`

**Exercise 6:** `f_ex.name(), f_ex.city()`

**Exercise 7:** `try_float(v)` for both blanks. The comprehension becomes:
```python
return [try_float(v) for v in values if try_float(v) is not None]
```

</details>

---
## Cumulative Review Exercises

These cover Days 1-3 (Pandas, NumPy, Data Cleaning). Keep those skills sharp!

In [ ]:
# Review 1 (Day 1 - Pandas): Filter a DataFrame using .loc
# Get all employees from Engineering with salary > 50000

df_rev = df_employees.copy()
high_eng = df_rev.loc[___, ["name", "department", "salary"]]

assert all(high_eng["department"] == "Engineering")
assert all(high_eng["salary"] > 50000)
print(f"Found {len(high_eng)} high-salary engineers")
print("Passed!")

In [ ]:
# Review 2 (Day 1 - Pandas): Use groupby to find the mean salary per department

dept_avg = df_rev.groupby(___)[___].mean().round(2)

assert isinstance(dept_avg, pd.Series)
assert len(dept_avg) == 4  # 4 departments
print(dept_avg)
print("Passed!")

In [ ]:
# Review 3 (Day 2 - NumPy): Create a 3x4 array of ones, then multiply by 7

arr = np.ones(___) * ___

assert arr.shape == (3, 4)
assert arr[0, 0] == 7.0
print(arr)
print("Passed!")

In [ ]:
# Review 4 (Day 2 - NumPy): Use boolean indexing to get values > 50
# from a numpy array

data = np.array([10, 55, 30, 72, 45, 88, 12, 60])
above_50 = data[___]

assert list(above_50) == [55, 72, 88, 60]
print("Above 50:", above_50)
print("Passed!")

In [ ]:
# Review 5 (Day 3 - Data Cleaning): Fill missing values with the column mean

df_dirty = pd.DataFrame({
    "score": [85, np.nan, 90, np.nan, 75, 80]
})
df_dirty["score"] = df_dirty["score"].fillna(df_dirty["score"].___())

assert df_dirty["score"].isna().sum() == 0
print("Filled scores:", df_dirty["score"].tolist())
print("Passed!")

In [ ]:
# Review 6 (Day 3 - Data Cleaning): Drop duplicate rows

df_dupes = pd.DataFrame({
    "name": ["Alice", "Bob", "Alice", "Charlie", "Bob"],
    "score": [90, 85, 90, 75, 85]
})
df_clean = df_dupes.___()

assert len(df_clean) == 3
print(df_clean)
print("Passed!")

In [ ]:
# Review 7 (Day 2 - NumPy): Calculate the dot product of two vectors

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
dot = np.___(a, b)

assert dot == 32  # 1*4 + 2*5 + 3*6
print(f"Dot product: {dot}")
print("Passed!")

In [ ]:
# Review 8 (Day 1 - Pandas): Convert a column to a different dtype

df_types = pd.DataFrame({"price": ["10.5", "20.3", "15.0"]})
df_types["price"] = df_types["price"].astype(___)

assert df_types["price"].dtype == np.float64
print("Dtype:", df_types["price"].dtype)
print("Passed!")

In [ ]:
# Review 9 (Day 3 - Data Cleaning): Parse a JSON string into a Python dict
import json

json_str = '{"name": "Deniz", "role": "data_scientist", "level": 1}'
parsed = json.___(json_str)

assert parsed["name"] == "Deniz"
assert parsed["role"] == "data_scientist"
print("Parsed:", parsed)
print("Passed!")

In [ ]:
# Review 10 (Day 2 - NumPy): Reshape a 1D array into a 2D matrix

flat = np.arange(12)  # [0, 1, 2, ..., 11]
matrix = flat.reshape(___, ___)

assert matrix.shape == (3, 4)
assert matrix[2, 3] == 11
print(matrix)
print("Passed!")

### Cumulative Review Solutions

<details><summary>Click to reveal all solutions</summary>

**Review 1:** `(df_rev["department"] == "Engineering") & (df_rev["salary"] > 50000)`

**Review 2:** `"department"` and `"salary"`

**Review 3:** `(3, 4)` and `7`

**Review 4:** `data > 50`

**Review 5:** `mean`

**Review 6:** `drop_duplicates`

**Review 7:** `dot`

**Review 8:** `float`

**Review 9:** `loads`

**Review 10:** `3, 4`

</details>

In [ ]:
# CHEAT SHEET - Day 4: Faker & Python Core
cheat = """
╔══════════════════════════════════════════════════════════════╗
║              FAKER & PYTHON CORE CHEAT SHEET                ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  FAKER                                                       ║
║  ─────                                                       ║
║  from faker import Faker                                     ║
║  fake = Faker()              # default (English)             ║
║  fake = Faker("tr_TR")       # Turkish locale                ║
║  Faker.seed(42)              # reproducible results          ║
║  fake.name()                 # random full name              ║
║  fake.email()                # random email                  ║
║  fake.date_between("-1y")    # random date in last year      ║
║  fake.random_int(1, 100)     # random integer                ║
║                                                              ║
║  NUMPY.RANDOM                                                ║
║  ────────────                                                ║
║  rng = np.random.default_rng(42)   # modern generator        ║
║  rng.normal(mean, std, size)       # bell curve              ║
║  rng.uniform(low, high, size)      # flat distribution       ║
║  rng.integers(low, high, size)     # random ints (excl high) ║
║  rng.choice(["A","B"], size)       # random picks            ║
║                                                              ║
║  COMPREHENSIONS                                              ║
║  ──────────────                                              ║
║  [x**2 for x in range(5)]           # list comp              ║
║  [x for x in lst if x > 0]          # with filter            ║
║  {k: v for k, v in pairs}           # dict comp              ║
║  {k: v for k,v in d.items() if v>0} # dict comp + filter     ║
║                                                              ║
║  FUNCTIONS                                                   ║
║  ─────────                                                   ║
║  def f(a, b=10):             # default argument              ║
║  def f(*args, **kwargs):     # flexible args                 ║
║  lambda x: x * 2             # anonymous function            ║
║                                                              ║
║  ERROR HANDLING                                              ║
║  ──────────────                                              ║
║  try:                                                        ║
║      risky_code()                                            ║
║  except ValueError as e:     # catch specific error          ║
║      handle(e)                                               ║
║  except (TypeError, KeyError):  # catch multiple             ║
║      handle()                                                ║
║  else:                       # runs if NO exception          ║
║      success()                                               ║
║  finally:                    # ALWAYS runs                   ║
║      cleanup()                                               ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""
print(cheat)

---
## Next up: Day 5 — PyTorchBasics

Tomorrow we get our hands on PyTorch: tensors, autograd, building a tiny neural network, and running a training loop. The fun stuff begins!